In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, box

# Task 1: Random points and buffers (100 m, 500 m, 1 km)

In [ ]:
np.random.seed(42)                      # fixed seed so the same points are generated every run

# 8 random points inside a 2 km x 2 km window (coordinates in metres, EPSG:32643 = UTM 43N)
x = np.random.uniform(0, 2000, 8)
y = np.random.uniform(0, 2000, 8)

points = gpd.GeoDataFrame(
    {"id": range(1, 9)},
    geometry=gpd.points_from_xy(x, y),
    crs="EPSG:32643"
)

# One GeoDataFrame per buffer distance
buf_100  = points.copy(); buf_100["geometry"]  = points.geometry.buffer(100)
buf_500  = points.copy(); buf_500["geometry"]  = points.geometry.buffer(500)
buf_1000 = points.copy(); buf_1000["geometry"] = points.geometry.buffer(1000)

# Plot the largest buffer first so the smaller ones stay visible on top
ax = buf_1000.plot(color="lightblue", edgecolor="blue", alpha=0.4, figsize=(8, 8))
buf_500.plot(ax=ax, color="khaki", edgecolor="orange", alpha=0.5)
buf_100.plot(ax=ax, color="salmon", edgecolor="red", alpha=0.6)
points.plot(ax=ax, color="black", markersize=20)

# Manual legend (a plain GeoDataFrame.plot does not create labels for each layer)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="lightblue", label="1 km"),
                   Patch(color="khaki", label="500 m"),
                   Patch(color="salmon", label="100 m")])
plt.title("Buffers of 100 m, 500 m and 1 km around random points")
plt.xlabel("Easting (m)"); plt.ylabel("Northing (m)")
plt.show()

# Task 2: Overlay of a circular and a rectangular polygon

In [ ]:
# Circle = buffer of a point, rectangle = box(minx, miny, maxx, maxy). They overlap in the middle.
circle = gpd.GeoDataFrame({"shape": ["circle"]},
                          geometry=[Point(300, 300).buffer(150)], crs="EPSG:32643")
rectangle = gpd.GeoDataFrame({"shape": ["rectangle"]},
                             geometry=[box(250, 200, 550, 450)], crs="EPSG:32643")

# Overlay with four different operations
operations = ["intersection", "union", "difference", "symmetric_difference"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, how in zip(axes.ravel(), operations):
    result = gpd.overlay(circle, rectangle, how=how)
    circle.boundary.plot(ax=ax, color="blue", linestyle="--")      # original circle outline
    rectangle.boundary.plot(ax=ax, color="red", linestyle="--")    # original rectangle outline
    result.plot(ax=ax, color="green", alpha=0.6, edgecolor="black")
    ax.set_title(f"overlay: {how}")
    ax.set_aspect("equal")

plt.suptitle("Circle (blue) and rectangle (red) overlay", fontsize=14)
plt.show()

# With the Given data

In [ ]:
import os

# Folder that contains buildings/, roads/, waterways/, pois/, landuse/ and nedumangad_boundary.shp
BASE = r"E:/IIST/1st sem/Scientific computing/Lab/Lab 6/nedumangad"
print("BASE exists:", os.path.isdir(BASE))

bp    = os.path.join(BASE, "buildings", "nedumangad_buildings.shp")
roads = os.path.join(BASE, "roads", "nedumangad_road.shp")
water = os.path.join(BASE, "waterways", "nedumangad_waterways.shp")
pois  = os.path.join(BASE, "pois", "nedumangad_pois.shp")
lulc  = os.path.join(BASE, "landuse", "nedumangad_landuse.shp")
bnd   = os.path.join(BASE, "nedumangad_boundary.shp")

In [ ]:
b = gpd.read_file(bp)
b.columns

In [ ]:
r = gpd.read_file(roads)
r.columns

In [ ]:
w = gpd.read_file(water)
w.columns

In [ ]:
p = gpd.read_file(pois)
p.columns

In [ ]:
l = gpd.read_file(lulc)
l.columns

In [ ]:
bd = gpd.read_file(bnd)
print(bd.crs)
print(bd.total_bounds)
bd.columns

In [ ]:
# CRS correction (preprocessing)
# Make sure to run this cell strictly once, if you run it multiple times bounds will be inf

b = b.set_crs("EPSG:4326", allow_override=True)   # rewrite the metadata
b = b.to_crs("EPSG:32643")                        # actually transform it

r = r.set_crs("EPSG:4326", allow_override=True)
r = r.to_crs("EPSG:32643")

w = w.set_crs("EPSG:4326", allow_override=True)
w = w.to_crs("EPSG:32643")

p = p.set_crs("EPSG:4326", allow_override=True)
p = p.to_crs("EPSG:32643")

l = l.set_crs("EPSG:4326", allow_override=True)
l = l.to_crs("EPSG:32643")

# Boundary has its own .prj, so just transform it (no override)
bd = bd.to_crs("EPSG:32643")

print(b.crs)
print(b.total_bounds)

print(w.crs)
print(w.total_bounds)

print(r.crs)
print(r.total_bounds)

print(p.crs)
print(p.total_bounds)

print(l.crs)
print(l.total_bounds)

print(bd.crs)
print(bd.total_bounds)   # should overlap the other layers' bounds

# Task 3: Data exploration and area of Nedumangad

In [ ]:
print("Total buildings:", len(b))
print("Total road segments:", len(r))
print("Total waterway segments:", len(w))
print("Total POIs:", len(p))
print("Total landuse polygons:", len(l))

print("\n--- LULC classes (count of polygons) ---")
print(l["fclass"].value_counts())
print("\nNumber of forest polygons:", (l["fclass"] == "forest").sum())

print("\n--- Road types ---")
print(r["fclass"].value_counts())
primary = r[r["fclass"] == "primary"]
print("\nPrimary road names:", list(primary["name"].dropna().unique()))

print("\n--- Waterway types ---")
print(w["fclass"].value_counts())

print("\n--- Top 10 POI types ---")
print(p["fclass"].value_counts().head(10))
print("\nNumber of hospitals:", (p["fclass"] == "hospital").sum())

In [ ]:
# Area of Nedumangad from the boundary layer (dissolve in case it has several polygons)
area_m2 = bd.union_all().area

print(f"Area of Nedumangad: {area_m2:,.0f} sq m")
print(f"                  = {area_m2/1e6:.2f} sq km")
print(f"                  = {area_m2/1e4:,.0f} hectares")

# Task 4: Buildings within 200 m of rivers and streams

In [ ]:
rs = w[w["fclass"].isin(["river", "stream"])]

# Buffer every river/stream by 200 m and merge into a single polygon
rs_buffer = gpd.GeoDataFrame(geometry=[rs.buffer(200).union_all()], crs="EPSG:32643")

# Keep only the buildings that touch the buffer
near_rs = gpd.sjoin(b, rs_buffer, predicate="intersects", how="inner")

print("Total buildings: ", len(b))
print("Buildings within 200 m of rivers/streams: ", len(near_rs))

named = near_rs["name"].dropna()
print("\nNames of buildings (10):")
print(named.head(10).to_string(index=False))

ax = rs_buffer.plot(color="lightblue", alpha=0.5, figsize=(9, 9))
b.plot(ax=ax, color="lightgrey")
near_rs.plot(ax=ax, facecolor="red", edgecolor="red", linewidth=2)
rs.plot(ax=ax, color="blue")
plt.title("Buildings within 200 m of rivers and streams"); plt.show()

# Task 5: Hospitals within 750 m of primary and trunk roads

In [ ]:
hospitals = p[p["fclass"] == "hospital"]
main_roads = r[r["fclass"].isin(["primary", "trunk"])]

road_buffer = gpd.GeoDataFrame(geometry=[main_roads.buffer(750).union_all()], crs="EPSG:32643")

near_roads = gpd.sjoin(hospitals, road_buffer, predicate="intersects", how="inner")

print("Total hospitals: ", len(hospitals))
print("Hospitals within 750 m of primary/trunk roads: ", len(near_roads))
print("\nHospital names (up to 10):")
print(near_roads["name"].dropna().head(10).to_string(index=False))

ax = road_buffer.plot(color="khaki", alpha=0.5, figsize=(9, 9))
main_roads.plot(ax=ax, color="orange", linewidth=1.5)
hospitals.plot(ax=ax, color="grey", markersize=30)
near_roads.plot(ax=ax, color="red", markersize=60)
plt.title("Hospitals within 750 m of primary and trunk roads"); plt.show()

# Task 6: Bridges on top of rivers

In [ ]:
bridges = r[r["bridge"] == "T"]                 # road segments flagged as bridge
rivers = w[w["fclass"] == "river"]              # add "stream" / "canal" here if you want them included

# Bridges that cross a river
on_river = gpd.sjoin(bridges, rivers[["geometry"]], predicate="intersects", how="inner")
on_river = on_river[~on_river.index.duplicated()]   # a bridge crossing 2 river pieces must be counted once

print("Total bridges on top of rivers: ", len(on_river))
print("Bridges without a name: ", on_river["name"].isna().sum())
print("\nBridge / road names:")
for n in on_river["name"].dropna().unique():
    print(" -", n)

ax = rivers.plot(linewidth=2, alpha=0.5, color="blue", figsize=(9, 9))
on_river.plot(ax=ax, color="red", linewidth=3)
plt.title("Bridges on top of rivers"); plt.show()

# Task 7: Primary and tertiary roads inside the forest

In [ ]:
forest = l[l["fclass"] == "forest"]
pt_roads = r[r["fclass"].isin(["primary", "tertiary"])]

# gpd.clip keeps only the part of each road that lies inside the forest polygons
roads_in_forest = gpd.clip(pt_roads, forest)

print("Road pieces inside forest: ", len(roads_in_forest))
print("\nNames of primary/tertiary roads inside forest:")
for n in roads_in_forest["name"].dropna().unique():
    print(" -", n)

ax = forest.plot(color="green", alpha=0.5, figsize=(9, 9))
pt_roads.plot(ax=ax, color="grey", linewidth=1)
roads_in_forest.plot(ax=ax, color="red", linewidth=2)
plt.title("Primary and tertiary roads inside forest"); plt.show()